# 02. Entailment dan Inference

Notebook ini membedakan **syntax**, **semantics**, dan **model**, lalu memakai ketiganya untuk memahami **logical entailment**. Kita akan mengenumerasi delapan kemungkinan dunia Wumpus, menentukan dunia yang sesuai dengan pengetahuan agent, dan melihat kapan sebuah kesimpulan benar-benar dijamin oleh knowledge base (KB).

Setelah menyelesaikan notebook ini, Anda diharapkan mampu:

1. membedakan bentuk kalimat, makna kalimat, dan model yang memenuhi kalimat;
2. menjelaskan $KB \vDash \alpha$ melalui relasi himpunan model;
3. memeriksa entailment dengan enumerasi model; dan
4. membedakan algoritma inferensi yang **sound** dan **complete**.

---
# 2.1 Syntax, Semantics, dan Model

## Penjelasan

| Istilah | Pertanyaan yang dijawab | Contoh |
|---|---|---|
| **Syntax** | Apakah susunan simbol ini merupakan kalimat yang valid? | `x + y = 4` *well-formed*, sedangkan `x3y+ =` tidak. |
| **Semantics** | Apa makna kalimat itu, dan kapan ia benar? | `x + y = 4` benar ketika nilai `x` dan `y` berjumlah 4. |
| **Model** | Pada keadaan dunia yang mana kalimat itu benar? | Penetapan `x = 2, y = 2` merupakan salah satu model dari `x + y = 4`. |

Syntax hanya memeriksa **bentuk**, bukan kebenaran. Sebuah sentence dapat tersusun secara valid tetapi bernilai salah pada suatu model. Semantics memberikan aturan untuk menentukan nilai kebenaran sentence pada setiap model. Jika sentence $\alpha$ benar pada model $m$, kita tulis

$$m \vDash \alpha,$$

dibaca "$m$ memenuhi $\alpha$". Himpunan seluruh model yang memenuhi $\alpha$ ditulis sebagai

$$M(\alpha) = \{m \mid m \vDash \alpha\}.$$

> **Catatan istilah:** model di sini adalah satu kemungkinan keadaan dunia atau interpretasi simbol, bukan model hasil pelatihan dalam *machine learning*.

## Logical entailment

Sentence $\alpha$ **meng-entail** sentence $\beta$, ditulis $\alpha \vDash \beta$, jika $\beta$ benar di **setiap** model tempat $\alpha$ benar. Definisi formalnya adalah

$$\boxed{\alpha \vDash \beta \quad\text{jika dan hanya jika}\quad M(\alpha) \subseteq M(\beta)}.$$

Ketika premisnya berupa knowledge base, notasinya menjadi $KB \vDash \alpha$. Kata pentingnya adalah **setiap**: satu contoh yang cocok belum cukup untuk membuktikan entailment, tetapi satu *counterexample*, model tempat KB benar dan kesimpulan salah, cukup untuk membantahnya.

---
# 2.2 Membangun Ruang Model Wumpus

## Penjelasan

Agent telah mengunjungi `[1,1]` tanpa merasakan *breeze*, lalu berpindah ke `[2,1]` dan merasakan *breeze*. Percept tersebut digabungkan dengan aturan Wumpus World di dalam KB. Agent ingin mengetahui isi tiga kotak yang belum diketahui: `[1,2]`, `[2,2]`, dan `[3,1]`.

![Tiga kotak yang isinya belum diketahui](img/slide-18-tiga-kotak-belum-diketahui.png)

Kita gunakan simbol $P_{x,y}$ yang berarti "ada pit di $[x,y]$". Informasi yang relevan dapat disederhanakan menjadi:

- tidak ada *breeze* di `[1,1]`, sehingga tidak ada pit di `[1,2]`: $\neg P_{1,2}$;
- ada *breeze* di `[2,1]`, sehingga sedikitnya salah satu dari `[2,2]` atau `[3,1]` berisi pit: $P_{2,2} \lor P_{3,1}$.

Kotak `[1,1]` dan `[2,1]` tidak perlu divariasikan: keduanya sudah dikunjungi dan agent masih hidup. Jadi, untuk tiga simbol yang belum diketahui, bentuk ringkas KB adalah

$$KB = \neg P_{1,2} \land (P_{2,2} \lor P_{3,1}).$$

Setiap simbol dapat bernilai `False` atau `True`, sehingga terdapat $2^3=8$ possible model.

Gambar berikut memperlihatkan delapan kombinasi tersebut. Tugas model checking adalah menyaring mana yang benar-benar konsisten dengan KB.

![Delapan possible model untuk tiga kotak](img/slide-19-delapan-model.png)

---
# 2.3 Menguji Dua Kesimpulan

## Penjelasan

Sekarang agent mempertimbangkan dua sentence:

- $\alpha_1 = \neg P_{1,2}$: "tidak ada pit di `[1,2]`";
- $\alpha_2 = \neg P_{2,2}$: "tidak ada pit di `[2,2]`".

![Figure 7.5: Relasi himpunan model KB, alpha 1, dan alpha 2](img/fig-7-5-model-alpha1-alpha2.png)

Pada Figure 7.5, seluruh $M(KB)$ berada di dalam $M(\alpha_1)$, sehingga $KB \vDash \alpha_1$. Sebaliknya, sebagian $M(KB)$ berada di luar $M(\alpha_2)$, sehingga $KB \nvDash \alpha_2$.

---
# 2.4 Logical Inference

## Penjelasan

**Entailment** adalah relasi matematis antara sentence: apakah kesimpulan benar di semua model premis. **Inference** adalah proses atau algoritma yang mencoba menurunkan kesimpulan tersebut. Notasi yang umum adalah

$$KB \vDash \alpha \quad\text{(entailment)}$$

$$KB \vdash_i \alpha \quad\text{(algoritma inferensi }i\text{ menurunkan }\alpha\text{)}.$$

Metode pada bagian sebelumnya disebut **model checking**: enumerasi semua possible model, pilih model yang memenuhi KB, lalu periksa apakah $\alpha$ benar pada semuanya.

![Figure 7.6: Hubungan dunia, representasi, dan logical reasoning](img/fig-7-6-logical-reasoning.png)

Sentence di dalam agent merepresentasikan keadaan dunia. Logical reasoning mengubah sentence lama menjadi sentence baru; jaminan formalnya memastikan bahwa representasi baru memang mengikuti dari pengetahuan sebelumnya.

## Sound dan complete

Dua sifat berikut menghubungkan hasil algoritma ($\vdash_i$) dengan kebenaran semantis ($\vDash$):

| Sifat | Jaminan formal | Makna praktis | Jika tidak dimiliki |
|---|---|---|---|
| **Sound** (*truth-preserving*) | Jika $KB \vdash_i \alpha$, maka $KB \vDash \alpha$ | Semua kesimpulan yang dikeluarkan memang dijamin oleh KB. | Algoritma dapat menghasilkan kesimpulan salah (*false positive*), misalnya menyebut kotak berpit sebagai aman. |
| **Complete** | Jika $KB \vDash \alpha$, maka $KB \vdash_i \alpha$ | Semua kesimpulan yang secara logis mengikuti pada akhirnya dapat ditemukan. | Algoritma dapat melewatkan kesimpulan benar (*false negative*), misalnya gagal mengenali kotak yang sebenarnya dapat dibuktikan aman. |

Algoritma yang sound tetapi tidak complete bersikap konservatif: jawabannya dapat kurang lengkap, tetapi yang berhasil disimpulkan tetap benar. Algoritma yang complete tetapi tidak sound dapat menemukan seluruh konsekuensi benar sekaligus mengeluarkan konsekuensi yang tidak valid. Untuk agent keselamatan-kritis, pelanggaran soundness biasanya lebih berbahaya karena agent dapat bertindak berdasarkan klaim yang keliru.

Model checking yang benar-benar memeriksa seluruh ruang model proposisional bersifat **sound dan complete**. Kekurangannya adalah biaya: dengan $n$ simbol proposisi terdapat $2^n$ model. Masalah pertumbuhan ini akan dibahas lebih lanjut pada Notebook 04.

---
# Ringkasan

- **Syntax** menentukan sentence mana yang tersusun valid; **semantics** menentukan artinya dan kapan sentence benar.
- **Model** adalah satu interpretasi atau kemungkinan dunia. $M(\alpha)$ adalah himpunan model yang memenuhi $\alpha$.
- $KB \vDash \alpha$ tepat ketika $M(KB) \subseteq M(\alpha)$, tidak ada model tempat KB benar tetapi $\alpha$ salah.
- `KB ⊭ α` hanya menyatakan bahwa $\alpha$ belum dijamin; itu tidak otomatis berarti $\alpha$ salah.
- **Soundness** mencegah kesimpulan tidak valid, sedangkan **completeness** menjamin tidak ada konsekuensi valid yang terlewat.
- Exhaustive model checking sound dan complete, tetapi ruang pencariannya tumbuh secara eksponensial.

---
# Latihan Soal

## Soal 1: Memahami ketidakpastian

Jelaskan perbedaan antara "$\alpha$ salah di semua model KB" dan "$\alpha$ tidak benar di semua model KB". Kaitkan jawaban Anda dengan $\alpha_2 = \neg P_{2,2}$ pada Bagian 2.3.

<details>
<summary><strong>Hint Soal 1</strong></summary>

"$\alpha$ salah di semua model KB" berarti $KB \vDash \neg\alpha$: KB menjamin negasinya. "$\alpha$ tidak benar di semua model KB" hanya berarti $KB \nvDash \alpha$: sedikitnya ada satu counterexample, tetapi mungkin juga ada model KB lain tempat $\alpha$ benar.

Untuk $\alpha_2$, satu model KB membuat $\alpha_2$ benar dan dua model KB membuatnya salah. Karena itu $KB \nvDash \alpha_2$ sekaligus $KB \nvDash \neg\alpha_2$. Isi `[2,2]` masih belum diketahui.

</details>

## Soal 2: Soundness dan completeness

Bandingkan dua algoritma inferensi berikut untuk agent Wumpus:

1. Algoritma A sound tetapi tidak complete.
2. Algoritma B complete tetapi tidak sound.

Apa yang dapat terjadi pada masing-masing agent? Mana yang umumnya lebih berbahaya jika hasil inferensi langsung digunakan untuk bergerak?

<details>
<summary><strong>Hint Soal 2</strong></summary>

Algoritma A tidak pernah menyatakan sesuatu yang tidak dijamin KB, tetapi dapat gagal menemukan kesimpulan yang valid. Agent mungkin terlalu ragu, melewatkan rute aman, atau membutuhkan eksplorasi tambahan.

Algoritma B dapat menemukan semua konsekuensi valid, tetapi juga dapat mengeluarkan kesimpulan yang tidak mengikuti dari KB. Agent bisa menganggap kotak berpit sebagai aman. Jika hasil digunakan langsung untuk bergerak, pelanggaran soundness pada B umumnya lebih berbahaya daripada ketidaklengkapan A.

</details>